# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook provides a step-by-step workflow for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. It guides you from data loading through exploratory data analysis (EDA) to making basic visualizations and insights, referencing all dataset structures by their `@id`s for reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL for the FAIR^2 dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
List available record sets, their field `@id`s, and column `@id`s (if present in schema). All names are referenced by their `@id` for clarity and reproducibility.

In [ ]:
# The dataset may contain multiple record sets defined by their @id
from pprint import pprint

print("Available record sets:")
if hasattr(metadata, 'record_sets'):
    for recset in metadata.record_sets:
        print(f"- @id: {recset['@id']}")
        if 'fields' in recset:
            print("  Fields by @id:")
            for field in recset['fields']:
                print(f"    - {field['@id']}")
            print()
        if 'columns' in recset:
            print("  Columns by @id:")
            for col in recset['columns']:
                print(f"    - {col['@id']}")
            print()
else:
    # For recent or minimal schemas, try the 'record_set' attribute (singular/plural handling)
    record_sets = getattr(metadata, 'record_set', None) or getattr(metadata, 'record_sets', None)
    if record_sets and len(record_sets) > 0:
        for rec in record_sets:
            rid = rec.get('@id', str(rec))
            print(f"- @id: {rid}")
            if 'fields' in rec:
                print('  Fields by @id:')
                for field in rec['fields']:
                    print(f"    - {field['@id']}")
            if 'columns' in rec:
                print('  Columns by @id:')
                for column in rec['columns']:
                    print(f"    - {column['@id']}")
    else:
        print("No record sets found in metadata. Try inspecting 'dataset.record_sets()' or 'ds.records()'.")

# For compact schemas, try listing record set IDs via the API
print("\nGetting record set @ids via API:")
try:
    record_sets = dataset.record_sets()
    for rs in record_sets:
        print(f"- {rs}")
except Exception as e:
    print(f"Unable to list record sets: {e}")

## 3. Data Extraction
Load data from each record set by referencing its `@id`. Each loaded record set is stored in a pandas DataFrame for analysis. All use of fields/columns/key attributes uses their `@id`s only.

In [ ]:
# Identify record set @ids
record_set_ids = []
try:
    # If the dataset supports record_sets() API
    record_set_ids = list(dataset.record_sets())
except Exception:
    # Otherwise, try extracting from metadata
    rs = getattr(metadata, 'record_set', None) or getattr(metadata, 'record_sets', None)
    if rs and isinstance(rs, list):
        for recset in rs:
            sid = recset['@id'] if isinstance(recset, dict) and '@id' in recset else str(recset)
            record_set_ids.append(sid)

print("Record set @ids for extraction:")
for rid in record_set_ids:
    print(f"- {rid}")

dataframes = {}
for recset_id in record_set_ids:
    print(f"\nExtracting records from record set: {recset_id}")
    try:
        records = list(dataset.records(record_set=recset_id))
        df = pd.DataFrame(records)
        print(f"  Loaded shape: {df.shape}")
        print("  Columns by @id:")
        pprint(df.columns.tolist())
        dataframes[recset_id] = df
    except Exception as e:
        print(f"  Error extracting {recset_id}: {e}")

# For demonstration: show the first 5 rows of the first loaded record set
if len(dataframes) > 0:
    first_recset_id = list(dataframes.keys())[0]
    print(f"\nPreview of record set {first_recset_id} (first 5 rows):")
    display(dataframes[first_recset_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps using only `@id` identifiers for numeric/group fields. Filter records by criteria, normalize a numeric field, and group by a key attribute for aggregation.

In [ ]:
# Select a record set and numeric field for analysis by @id
# First, print available record sets and numeric fields:
print("\nAvailable DataFrames (record set @ids):", list(dataframes.keys()))
if len(dataframes) == 0:
    raise Exception("No data loaded. Please review previous steps.")

selected_recset_id = list(dataframes.keys())[0]
df = dataframes[selected_recset_id]
print(f"\nAvailable columns for {selected_recset_id} (by @id):\n", df.columns.tolist())

# Try to identify a numeric field by @id -- for demo, pick the first float/int column
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break

if numeric_field_id is None:
    print("No numeric field found to demonstrate filtering. Skipping EDA block.")
else:
    print(f"Selected numeric field for EDA: {numeric_field_id}")
    threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0

    # Filter records
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records where {numeric_field_id} > {threshold:.2f}")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std(ddof=0)
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Attempt grouping by a non-numeric field
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col]):
            group_field_id = col
            break
    if group_field_id:
        print(f"\nGrouping by field: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(grouped_df.head())
    else:
        print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize a data distribution or a simple relationship, using columns referenced by their `@id` only.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Use EDA results if available; otherwise, plot from any numeric column in any loaded DataFrame
if 'numeric_field_id' in locals() and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()
else:
    # Plot all numeric columns
    for recset_id, dframe in dataframes.items():
        num_cols = [c for c in dframe.columns if pd.api.types.is_numeric_dtype(dframe[c])]
        for num_col in num_cols:
            plt.figure(figsize=(7, 4))
            sns.histplot(dframe[num_col].dropna(), kde=True)
            plt.title(f"Distribution of {num_col} in {recset_id}")
            plt.xlabel(num_col)
            plt.ylabel('Frequency')
            plt.show()

## 6. Conclusion
This notebook demonstrated loading, exploring, and analyzing the FAIR^2 dataset on ordered logistic regression results for knowledge adoption in Northern Kenya using the `mlcroissant` library. Key dataset structures were referenced by `@id` throughout to ensure reproducibility. More advanced analysis can be built upon this workflow, including modeling, domain-specific exploration, or integration with FAIR data repositories.